# Fine-tunning on Figure 3D MethylBERT data with EpigenDnabert2

This tutorial demonstrates:
1. **How to perform fine-tunning on the prepared data and save results to the local directory**.
---

### Step 0: Import needed classes and set paths

In [1]:
from methyldl.modelling.dnabert2 import EpigenDnabert2, TrainingArguments
from methyldl.data.dataset import SupervisedDataset
import pandas as pd
# data_path = "../Data/Curated/MethylBERT_Figure3D"
# data_path = "../Data/Curated/MethylBERT_Figure3D_customsplit"
# data_path = "../Data/Curated/rrms_dmrs_only_mincov40/chr2"
# data_path = "../Data/Curated/rrms_dmrs_only_mincov40/chr1_cpg_counts_selected"
# data_path = "../Data/Curated/rrms_dmrs_only_mincov40/chr8_dmr_cuts"
# data_path = "../Data/Curated/rrms_dmrs_only_mincov40_corrected_imporvement_experiments/chr19_top3000_global_dmrs_dmrs_cuts"
# data_path = "../Data/Curated/experiment_c/chr19"
# data_path = "../../../Data/SyntethicMethylDLTestData/"
data_path = '/home/luna.kuleuven.be/u0169940/Data/Loyfer/TrainingDataWithRejectionDMRsStratified/'
from torch import nn
import os

/home/luna.kuleuven.be/u0169940/.cache/pypoetry/virtualenvs/methyldl-GStZGe-R-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Inspecting data in path 

### Step 1: Perform fine-tunning

In [2]:
dmrs = pd.DataFrame(pd.concat([pd.read_parquet(data_path+f"/{split}.parquet") for split in ["train", "valid", "test"]])["dmr_label"].unique())
dmrs.columns  = ["dmr_label"]

In [ ]:
model_instance = EpigenDnabert2(use_cpg_methylation=True, max_sequence_length=150, 
                                trust_remote_code=True, 
                                foundation_model_huggingface="../foundationalModels/DNABERT-2-117M",
                                num_labels=40,
                                num_dmr_labels=max(dmrs["dmr_label"])+1)

training_args = TrainingArguments(
            run_name = "dnabert2_with_dmr_head_loyfer",
            per_device_train_batch_size = 512,
            per_device_eval_batch_size = 3200,
            gradient_accumulation_steps = 1,
            learning_rate = 0.0004,
            fp16 = True,
            save_steps = 200,
            output_dir ="DNABERT2_with_dmr_head_loyfer_atlas_with_reject_dmr_stratified_projected_inputs",
            eval_strategy = "steps",
            eval_steps = 200, 
            warmup_steps = 100, 
            logging_steps = 100, 
            num_train_epochs = 100, 
            overwrite_output_dir = True, 
            log_level = "info",
            find_unused_parameters = False,
            batch_eval_metrics = False,
            eval_and_save_results = True,
            remove_unused_columns=False,
            eval_accumulation_steps = 8,
            torch_empty_cache_steps = 10,
            prediction_loss_only=False,
            gradient_checkpointing=False,
            skip_memory_metrics=True,
            auto_find_batch_size=False,
            max_grad_norm = 1.0,
            adam_beta1=0.9,
            adam_beta2=0.98,
            adam_epsilon=1e-6
            )

/home/luna.kuleuven.be/u0169940/.cache/huggingface/modules/transformers_modules/DNABERT-2-117M/bert_layers.py:135: UserWarning: Using triton implementation of flash attention 2
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ../foundationalModels/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ../foundationalModels/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/luna.kuleuven.be/u0169940/Repos/methyldl/methyldl/modelling/dnabert2

In [ ]:
# model_instance.model.classifier = nn.Linear(768,out_features=3,bias=True)
# model_instance.num_labels=3
# model_instance.model.num_labels = 3
# model_instance.model.config.problem_type = "single_label_classification"

In [4]:
train_dataset = SupervisedDataset(tokenizer=model_instance.tokenizer, 
                                        data_path_or_list=os.path.join(data_path, "train"), 
                                        kmer=-1,data_interface="pandas",
                                        lazy_tokenization = True,
                                        include_dmr_ids=True)

val_dataset = SupervisedDataset(tokenizer=model_instance.tokenizer, 
                                        data_path_or_list=os.path.join(data_path, "valid"), 
                                        kmer=-1,data_interface="pandas",
                                        lazy_tokenization = True,
                                        include_dmr_ids=True)

In [6]:
import pandas as pd

In [7]:
model_instance.fine_tune(train_dataset=train_dataset, 
                         val_dataset=val_dataset,
                         test_dataset="empty",
                         training_args=training_args, data_interface="pandas")

/home/luna.kuleuven.be/u0169940/Repos/methyldl/methyldl/modelling/dnabert2.py:515: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  return transformers.Trainer(model=self.model,
Using auto half precision backend
***** Running training *****
  Num examples = 256,836
  Num Epochs = 100
  Instantaneous batch size per device = 512
  Total train batch size (w. parallel, distributed & accumulation) = 512
  Gradient Accumulation steps = 1
  Total optimization steps = 50,200
  Number of trainable parameters = 121,657,768


Starting to initialize datasets
Train is initialized
Val is initialized
All datasets are successfully initiated


Step,Training Loss,Validation Loss
200,2.459000,2.510840



***** Running Evaluation *****
  Num examples = 117064
  Batch size = 3200
Saving model checkpoint to DNABERT2_with_dmr_head_loyfer_atlas_with_reject_dmr_stratified/checkpoint-200
Configuration saved in DNABERT2_with_dmr_head_loyfer_atlas_with_reject_dmr_stratified/checkpoint-200/config.json
Model weights saved in DNABERT2_with_dmr_head_loyfer_atlas_with_reject_dmr_stratified/checkpoint-200/model.safetensors
tokenizer config file saved in DNABERT2_with_dmr_head_loyfer_atlas_with_reject_dmr_stratified/checkpoint-200/tokenizer_config.json
Special tokens file saved in DNABERT2_with_dmr_head_loyfer_atlas_with_reject_dmr_stratified/checkpoint-200/special_tokens_map.json


KeyboardInterrupt: 

In [ ]:
os.listdir('DNABERT2_with_dmr_head_loyfer_atlas_with_reject_dmr_stratified')

In [ ]:
model_instance = EpigenDnabert2(use_cpg_methylation=True, max_sequence_length=150, 
                                trust_remote_code=True, 
                                foundation_model_huggingface="../foundationalModels/DNABERT-2-117M",
                                num_labels=40,
                                num_dmr_labels=max(dmrs["dmr_label"])+1,
                                fine_tuned_model_path="DNABERT2_with_dmr_head_loyfer_atlas_with_reject_dmr_stratified/checkpoint-50200/model.safetensors")

In [ ]:
test_dataset = SupervisedDataset(tokenizer=model_instance.tokenizer, 
                                        data_path_or_list=os.path.join(data_path, "test"), 
                                        kmer=-1,data_interface="pandas",
                                        lazy_tokenization = True,
                                        include_dmr_ids=True)

In [ ]:
res = model_instance.predict(test_dataset, batch_size=3200)

In [ ]:
import pickle
import numpy as np

In [ ]:
with open("DNABERT2_with_dmr_head_loyfer_atlas_with_reject_dmr_stratified/predictions_test.pkl", "wb") as f:
    pickle.dump(res, f)

In [ ]:
predictions = res[0].argmax(axis=1)
labels = res[1]